# Sea-surface density (σ₀) over the DBO, through time

Combines satellite **sea-surface temperature** (MODIS Aqua, streamed) and the **already-cached
sea-surface salinity** (OISSS v3 multimission — Aquarius + SMAP + SMOS optimally interpolated) on a
shared 4-km EASE-Grid 2.0 North grid to compute monthly **surface potential density anomaly, σ₀**
(kg m⁻³) over the Distributed Biological Observatory (DBO 1–8), and to ask whether the surface is
getting lighter (more buoyant / more stratified) — and whether that's driven more by **freshening**
or by **warming**.

### What it does
1. Streams monthly MODIS Aqua nighttime SST cloud-natively from NASA Earthdata.
2. Reprojects SST to the DBO EASE-Grid 2.0 North 4-km grid (equal-area bucket average — the same
   single-gridding used in `1_pull_data/06_modis-sst-daily.ipynb`).
3. **Loads the cached monthly OISSS SSS stack**
   (`salinity/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc`) — already streamed and
   Gaussian-gridded to the **same** EASE2 grid by
   `2_figures/figS04_oisss-sss-box-trends.ipynb`, so no SSS re-streaming is needed.
4. Combines them with the **TEOS-10** equation of state (`gsw`) to get σ₀ per pixel, per month.
5. Builds per-DBO-box σ₀ time series, fits a robust trend (Theil-Sen + AR(1)-corrected
   Mann-Kendall), and **decomposes the density trend into thermal vs haline contributions**.

### Two things to be clear about
- **Pressure.** The "pressure" in the seawater equation of state is *water* pressure (depth, in
  dbar), which at the sea surface is **0 by definition** — not the *atmospheric* sea-level
  pressure from MERRA-2 (that is air pressure, a different physical quantity, and is not used
  here). We pass `p = 0 dbar` throughout.
- **Record length.** OISSS v3 monthly begins **September 2011**, so σ₀ can go back to 2011 (MODIS
  SST reaches 2002, so SST is not the limit). *Earlier* SMAP-only runs were capped at April 2015;
  switching to OISSS extends the record by ~3.5 years.
- **Surface density ≠ stratification.** Satellites see only the surface. A falling σ₀ means the
  surface is getting more buoyant (a strong *indicator* of strengthening stratification), but the
  full stratification claim needs subsurface density (Argo/CTD).

## Imports

In [5]:
import warnings, time as _time
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, xarray as xr
import dask.array as dka
import earthaccess
from pyresample import AreaDefinition
from pyresample.bucket import BucketResampler
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from scipy.stats import theilslopes, kendalltau, norm
import gsw
try:
    import cmocean
    DENSCMAP = cmocean.cm.dense
except ImportError:
    DENSCMAP = plt.get_cmap("viridis")
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## Configuration

Only the SST product is streamed; SSS is read from the cached OISSS stack that
`2_figures/figS04_oisss-sss-box-trends.ipynb` already built. Confirm the SST short name
with the search cell below the first time you run.

In [6]:
START_DATE = date(2011, 9, 1)     # OISSS v3 monthly record start (2011-09) — sets the density start
END_DATE   = date.today()

DBO_BBOX = (-180.0, 50.0, -124.0, 74.0)     # lon0, lat0, lon1, lat1

# --- SST product (streamed); confirm short name via the search cell below ---
SST_SHORT = "MODIS_AQUA_L3_SST_THERMAL_MONTHLY_4KM_NIGHTTIME_V2019.0"
SST_VAR   = "sst"

# --- cached SSS stack (NOT streamed) — OISSS v3 multimission monthly, built by
#     2_figures/figS04_oisss-sss-box-trends.ipynb (section 4). Replaces the old SMAP cache.
#     Path is relative to where this notebook runs (project root); edit if needed.
SSS_CACHE = Path("salinity") / "outputs_longterm_oisss" / "monthly_sss_ease2_oisss_gauss.nc"

# --- shared analysis grid: EASE-Grid 2.0 North 4 km subset (identical to SST + SSS notebooks) ---
RESOLUTION = "4km"
dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))
xs, ys = dbo_area.get_proj_vectors()
lons2d, lats2d = dbo_area.get_lonlats()

OUT_DIR = Path("outputs_density"); OUT_DIR.mkdir(parents=True, exist_ok=True)

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
print(f"record window: {START_DATE} -> {END_DATE} | grid {RESOLUTION} EASE-2 North {dbo_area.shape}")
print(f"SSS cache: {SSS_CACHE}  (exists: {SSS_CACHE.exists()})")

record window: 2011-09-01 -> 2026-07-29 | grid 4km EASE-2 North (845, 906)
SSS cache: salinity/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc  (exists: False)


## Earthdata authentication

In [7]:
auth = earthaccess.login(persist=True)

## Helpers — search, subset, and the SST regridder

SST is reprojected with an equal-area **bucket average** (`BucketResampler`): the 4-km source
pixels are binned into the 4-km EASE2 cells — a resolution-preserving single grid, matching the
SSS gridding so the two line up cell-for-cell.

In [8]:
def _search(sn, tspan):
    for k in range(5):
        try:
            return earthaccess.search_data(short_name=sn, temporal=tspan)
        except Exception:
            _time.sleep(2*(k+1))
    raise RuntimeError(f"CMR search failed for {sn}")

def _gran_start(g):
    te = g["umm"]["TemporalExtent"]
    return te["RangeDateTime"]["BeginningDateTime"] if "RangeDateTime" in te else te["SingleDateTime"]

def _month_of(g):
    return np.datetime64(_gran_start(g)[:7] + "-01")     # normalise to month start

def _bbox_sel(da, bbox, latn, lonn):
    lon0,lat0,lon1,lat1 = bbox
    la, lo = da[latn].values, da[lonn].values
    ls = slice(lat1,lat0) if la[0] > la[-1] else slice(lat0,lat1)
    xs_ = slice(lon0,lon1) if lo[0] < lo[-1] else slice(lon1,lon0)
    return da.sel({latn: ls, lonn: xs_})

def regrid_sst(da):
    sub = _bbox_sel(da.squeeze(), DBO_BBOX, "lat", "lon")
    lon, lat = sub["lon"].values, sub["lat"].values
    vals = sub.values.astype(float)
    vals = np.where(np.isfinite(vals) & (vals > -2.0) & (vals < 40.0), vals, np.nan)  # physical SST range
    lon2d, lat2d = np.meshgrid(lon, lat)
    resampler = BucketResampler(dbo_area, dka.from_array(lon2d, chunks="auto"),
                                          dka.from_array(lat2d, chunks="auto"))
    out = resampler.get_average(dka.from_array(vals, chunks="auto"))
    return (out.compute() if hasattr(out, "compute") else out).astype(np.float32)

## Stream + reproject monthly SST to an EASE2 stack
One granule per month is binned to the shared grid; the stack is cached as NetCDF (delete the
cache file to force a rebuild).

In [9]:
def build_sst_stack(cache_name="monthly_sst_ease2.nc"):
    cache = OUT_DIR / cache_name
    if cache.exists():
        da = xr.open_dataarray(cache)
        print(f"loaded {cache.name}: {da.sizes['time']} months "
              f"({str(da.time.values[0])[:7]} -> {str(da.time.values[-1])[:7]})")
        return da
    gr = _search(SST_SHORT, (START_DATE.isoformat(), END_DATE.isoformat()))
    by_month = {}
    for g in gr:
        by_month.setdefault(_month_of(g), g)             # one granule per month
    months = sorted(by_month)
    print(f"{SST_SHORT}: {len(months)} monthly granules {str(months[0])[:7]} -> {str(months[-1])[:7]}")
    ny, nx = dbo_area.shape
    buf = np.full((len(months), ny, nx), np.nan, np.float32)
    times = []
    t0 = _time.time()
    for k, m in enumerate(months):
        try:
            f = earthaccess.open([by_month[m]], show_progress=False)[0]
            with xr.open_dataset(f, engine="h5netcdf", chunks={}) as ds:
                v = SST_VAR if SST_VAR in ds.variables else list(ds.data_vars)[0]
                buf[len(times)] = regrid_sst(ds[v])
            times.append(np.datetime64(m))
        except Exception as e:
            print(f"  skip {str(m)[:7]} ({type(e).__name__}: {str(e)[:60]})")
            continue
        if k % 12 == 0:
            print(f"  {k}/{len(months)} months  ({_time.time()-t0:.0f}s)")
    da = xr.DataArray(buf[:len(times)], dims=("time","y","x"),
                      coords={"time": np.array(times), "y": ys, "x": xs}, name="sst",
                      attrs={"product": SST_SHORT, "grid": "EASE-Grid 2.0 North (EPSG:6931) 4 km"})
    da.to_netcdf(cache, encoding={"sst": {"zlib": True, "complevel": 4}})
    print(f"built + cached {da.sizes['time']} months -> {cache.name} ({(_time.time()-t0)/60:.1f} min)")
    return da

sst = build_sst_stack()

MODIS_AQUA_L3_SST_THERMAL_MONTHLY_4KM_NIGHTTIME_V2019.0: 177 monthly granules 2011-08 -> 2026-05
  0/177 months  (9s)
  12/177 months  (77s)
  24/177 months  (146s)
  36/177 months  (211s)
  48/177 months  (281s)
  60/177 months  (357s)
  72/177 months  (440s)
  84/177 months  (515s)


## Load the cached monthly OISSS SSS stack
Already gridded to the same EASE2 grid by `2_figures/figS04_oisss-sss-box-trends.ipynb`
(section 4). OISSS `sss` is practical salinity (numerically PSU), which is what `gsw.SA_from_SP`
expects below. We verify the grid matches before using it.

In [8]:
from pathlib import Path

# OISSS v3 multimission monthly SSS stack (var 'sss', dims (time,y,x)), built by section 4 of
# 2_figures/figS04_oisss-sss-box-trends.ipynb. It lives in salinity/outputs_longterm_oisss/.
# This REPLACES the old SMAP-derived monthly_sss_ease2_*.nc cache. OISSS 'sss' is practical
# salinity (numerically PSU) — exactly what gsw.SA_from_SP expects in the σ₀ cell below.
SSS_CACHE = None                       # set to a path string to force a specific file

if SSS_CACHE is None:
    _dirs = [Path.cwd(), Path.cwd() / "salinity", Path.cwd().parent,
             Path.cwd().parent / "salinity", Path.home() / "ADD_YOUR_PATH" / "salinity"]  # add your path
    _hits = []
    for b in _dirs:
        _hits += sorted((b / "outputs_longterm_oisss").glob("monthly_sss_ease2_oisss_*.nc"))
    _hits = list(dict.fromkeys(_hits))
    _hits = [p for p in _hits if "gauss" in p.name] or _hits   # prefer the NaN-safe gauss regrid
    if not _hits:
        raise FileNotFoundError(
            "monthly_sss_ease2_oisss_*.nc not found. Run section 4 of "
            "2_figures/figS04_oisss-sss-box-trends.ipynb (streams + caches the OISSS stack), "
            f"or set SSS_CACHE manually. Searched: {[str(b/'outputs_longterm_oisss') for b in _dirs]}")
    SSS_CACHE = _hits[0]
else:
    SSS_CACHE = Path(SSS_CACHE)
    if not SSS_CACHE.exists():
        raise FileNotFoundError(f"SSS_CACHE set but missing: {SSS_CACHE}")

sss = xr.open_dataarray(SSS_CACHE)
assert (sss.sizes["y"], sss.sizes["x"]) == dbo_area.shape, \
    f"SSS grid {(sss.sizes['y'], sss.sizes['x'])} != grid {dbo_area.shape}"
print(f"loaded OISSS SSS: {SSS_CACHE}\n  {sss.sizes['time']} months "
      f"({str(sss.time.values[0])[:7]} -> {str(sss.time.values[-1])[:7]})  "
      f"product={sss.attrs.get('product','?')}, resample={sss.attrs.get('resample','?')}")

loaded OISSS SSS: ~/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc
  168 months (2011-09 -> 2025-08)  product=OISSS_L4_multimission_monthly_v3, resample=gauss


## Compute σ₀ with TEOS-10

Align SST and SSS on their common months (both dated to month-start), then convert practical
salinity → absolute salinity and in-situ temperature → conservative temperature at the surface
(`p = 0 dbar`), and take σ₀ = ρ(SA, CT, 0) − 1000.

In [9]:
# normalise both time axes to month-start so they align cleanly regardless of datetime unit
sst = sst.assign_coords(time=pd.to_datetime(sst.time.values).to_period("M").to_timestamp())
sss = sss.assign_coords(time=pd.to_datetime(sss.time.values).to_period("M").to_timestamp())
common = sst.time.to_index().intersection(sss.time.to_index())
if len(common) == 0:
    raise RuntimeError("No overlapping months between SST and SSS — check the two time ranges.")
sst_c = sst.sel(time=common).values
sss_c = sss.sel(time=common).values
print(f"{len(common)} months with both SST and SSS ({common[0]:%Y-%m} -> {common[-1]:%Y-%m})")

# lons2d/lats2d are (y,x); they broadcast against the (time,y,x) fields.
SA   = gsw.SA_from_SP(sss_c, 0.0, lons2d, lats2d)   # practical -> absolute salinity
CT   = gsw.CT_from_t(SA, sst_c, 0.0)                # in-situ -> conservative temperature
sig0 = gsw.sigma0(SA, CT).astype(np.float32)        # surface potential density anomaly (kg/m^3)

ctime = common.values
density = xr.DataArray(
    sig0, dims=("time","y","x"), coords={"time": ctime, "y": ys, "x": xs}, name="sigma0",
    attrs={"units": "kg m-3",
           "long_name": "surface potential density anomaly (TEOS-10, p=0 dbar)",
           "inputs": f"SST={SST_SHORT}; SSS={sss.attrs.get('product', SSS_CACHE.name)}",
           "grid": "EASE-Grid 2.0 North (EPSG:6931) 4 km",
           "note": "atmospheric pressure NOT used; surface water pressure p=0 dbar"})
density.to_netcdf(OUT_DIR / "sea_surface_density_sigma0_ease2_monthly.nc")
print(f"saved σ₀ stack  (range {np.nanmin(sig0):.2f} .. {np.nanmax(sig0):.2f} kg/m³)")

125 months with both SST and SSS (2015-03 -> 2025-08)
saved σ₀ stack  (range 9.85 .. 26.88 kg/m³)


---

# Part B — backfill the SST stack to 2011 and fit the per-pixel density trend (Fig. 6a input)

*This part was a separate notebook before the repository was consolidated; it starts from its own imports and configuration, so it can be run on its own after the sections above.*

# Sea-surface density (σ₀) trend — **map** version, EASE-Grid 2.0 with the DBO boxes

The companion to the per-DBO-box σ₀ trend panels in `1_pull_data/08_sea-surface-density.ipynb`. Same
inputs (MODIS Aqua nighttime SST + OISSS v3 multimission SSS, TEOS-10 σ₀ at p = 0 dbar, 4-km
EASE-Grid 2.0 North), but the trend is fit **per pixel** instead of per box, and drawn as a map in
the same style as the MODIS SST trend map (`1_pull_data/06_modis-sst-daily.ipynb` §15).

### What it makes
1. **σ₀ trend map** — Theil-Sen slope in **kg m⁻³ per decade**, hatched where the AR(1)-adjusted
   Mann-Kendall p < 0.1. Coral = surface getting **lighter / more buoyant**, blue = **denser** —
   the same colour convention as the box-panel figure.
2. **Coverage map** — how many years each pixel actually has, so it is obvious where the map is
   supported by data and where it is not.
3. **Thermal vs. haline decomposition maps** — the same α/β split the box figure does, per pixel:
   how much of the density trend is the temperature trend and how much is the salinity trend.
4. **Consistency table** — box-averaged map slopes vs. the box time-series slopes, so the map and
   the line figure can be checked against each other.

### Units, not percent
The SST trend map is drawn in **%/decade vs. the pixel mean**. That does not transfer to σ₀: σ₀ is
an *anomaly* (density − 1000 kg m⁻³) whose zero point is arbitrary, so a percentage of it is not a
meaningful quantity. This map is therefore in **absolute kg m⁻³ per decade**, the same units as the
slope annotations on the box panels.

### The salinity coverage caveat (read this)
OISSS is an optimal interpolation of L-band radiometer SSS (Aquarius/SMAP/SMOS). It is retrieved
only over **open water**, is flagged near land and near the ice edge, and its record starts
**2011-09**. So σ₀ exists only where and when salinity exists: expect the map to fade out under the
seasonal ice cover in the northern boxes (DBO 4–8), along the coasts, and in the Canadian
Archipelago. That is why the composites below are restricted to the **May–September** open-water
season and why every pixel is required to have a minimum number of years before it is fit — the
coverage panel shows exactly what survived.

## Imports

In [1]:
import warnings, time as _time
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
from pyresample import AreaDefinition
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs, cartopy.feature as cfeature
from scipy.stats import norm, theilslopes, kendalltau
import gsw
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## Configuration

The grid, bounding box and DBO polygons are copied verbatim from `08_sea-surface-density.ipynb` so
this notebook lands on exactly the same 4-km EASE-2 cells.

In [2]:
# --- shared analysis grid: EASE-Grid 2.0 North 4 km subset (identical to the SST/SSS/σ₀ notebooks) ---
dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))
xs, ys = dbo_area.get_proj_vectors()
lons2d, lats2d = dbo_area.get_lonlats()

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}

# --- analysis choices ---
GROW_MONTHS  = [5, 6, 7, 8, 9]   # open-water season; OISSS is unusable under ice
ANNUAL_STAT  = "median"          # "median" (robust, matches the SST trend map) or "mean"
MIN_SEASON_MONTHS = 3            # a YEAR needs this many valid months in the window to count
MIN_YEARS    = 8                 # a pixel needs this many annual values to be fit
P_HATCH      = 0.10              # hatch pixels with AR(1)-adjusted MK p below this

# --- palette: same convention as the box-panel figure ---
LIGHTEN = "#EE6C4D"   # σ₀ decreasing -> lighter / more buoyant (coral)
DENSIFY = "#3D5A80"   # σ₀ increasing -> denser (dusty blue)
STYLE = {"font.family": "DejaVu Sans", "font.size": 9,
         "axes.titlesize": 10.5, "axes.labelsize": 9,
         "figure.facecolor": "white", "axes.facecolor": "white"}

OUT_DIR = Path("outputs_density"); OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"grid {dbo_area.shape} | season {GROW_MONTHS} | annual stat = {ANNUAL_STAT} | "
      f"min {MIN_SEASON_MONTHS} months/year | min {MIN_YEARS} years/pixel")

grid (845, 906) | season [5, 6, 7, 8, 9] | annual stat = median | min 3 months/year | min 8 years/pixel


## Backfill the monthly SST stack to the OISSS start (2011-09)

Extends the `monthly_sst_ease2.nc` cache written by `08_sea-surface-density.ipynb` back to September 2011 so the density record covers the full OISSS period.

In [12]:
from datetime import date

RUN_BACKFILL   = True                                 # set False once the extended cache exists
BACKFILL_START = date(2011, 9, 1)                     # OISSS v3 monthly record start

# same product + bbox the original cache was built from (08_sea-surface-density.ipynb)
SST_SHORT = "MODIS_AQUA_L3_SST_THERMAL_MONTHLY_4KM_NIGHTTIME_V2019.0"
SST_VAR   = "sst"
DBO_BBOX  = (-180.0, 50.0, -124.0, 74.0)              # lon0, lat0, lon1, lat1

# existing cache (input, never modified) and the extended one this section writes
BASE_SST = None            # None -> search; or set a path string
EXT_SST  = None            # None -> BASE_SST's folder / monthly_sst_ease2_2011on.nc

_cands = [Path.cwd() / "outputs_density" / "monthly_sst_ease2.nc",
          Path.cwd() / "density" / "outputs_density" / "monthly_sst_ease2.nc",
          Path.home() / "outputs_density" / "monthly_sst_ease2.nc"]  # add your path
BASE_SST = Path(BASE_SST) if BASE_SST else next((p for p in _cands if p.exists()), None)
if BASE_SST is None:
    print("no existing monthly_sst_ease2.nc found — set BASE_SST explicitly, or skip this section "
          "and let 08_sea-surface-density.ipynb build the SST stack from scratch")
else:
    EXT_SST = Path(EXT_SST) if EXT_SST else BASE_SST.with_name("monthly_sst_ease2_2011on.nc")
    _b = xr.open_dataarray(BASE_SST)
    _t = pd.to_datetime(_b.time.values)
    print(f"existing cache : {BASE_SST}\n  {len(_t)} months ({_t[0]:%Y-%m} -> {_t[-1]:%Y-%m})")
    print(f"extended cache : {EXT_SST}  (exists: {EXT_SST.exists()})")
    _need = pd.period_range(BACKFILL_START, _t[0] - pd.DateOffset(months=1), freq="M")
    print(f"months to backfill: {len(_need)}"
          + (f"  ({_need[0]} -> {_need[-1]})" if len(_need) else "  — nothing to do"))
    _b.close()

existing cache : ~/outputs_density/monthly_sst_ease2.nc
  133 months (2015-03 -> 2026-04)
extended cache : ~/outputs_density/monthly_sst_ease2_2011on.nc  (exists: False)
months to backfill: 42  (2011-09 -> 2015-02)


In [13]:
if not RUN_BACKFILL or BASE_SST is None:
    print("skipped (RUN_BACKFILL is False, or no base cache)")
elif EXT_SST.exists():
    _e = xr.open_dataarray(EXT_SST); _et = pd.to_datetime(_e.time.values)
    print(f"{EXT_SST.name} already exists: {len(_et)} months ({_et[0]:%Y-%m} -> {_et[-1]:%Y-%m}) "
          "— delete it to force a rebuild"); _e.close()
elif len(_need) == 0:
    print("nothing to backfill — the existing cache already reaches BACKFILL_START")
else:
    import time as _time
    import earthaccess, dask.array as dka
    from pyresample.bucket import BucketResampler

    auth = earthaccess.login(persist=True)

    def _gran_start(g):
        te = g["umm"]["TemporalExtent"]
        return te["RangeDateTime"]["BeginningDateTime"] if "RangeDateTime" in te else te["SingleDateTime"]

    def _bbox_sel(da, bbox, latn, lonn):
        lon0, lat0, lon1, lat1 = bbox
        la, lo = da[latn].values, da[lonn].values
        ls  = slice(lat1, lat0) if la[0] > la[-1] else slice(lat0, lat1)
        xs_ = slice(lon0, lon1) if lo[0] < lo[-1] else slice(lon1, lon0)
        return da.sel({latn: ls, lonn: xs_})

    def regrid_sst(da):
        """Equal-area bucket average onto dbo_area — identical to 08_sea-surface-density.ipynb."""
        sub = _bbox_sel(da.squeeze(), DBO_BBOX, "lat", "lon")
        lon, lat = sub["lon"].values, sub["lat"].values
        vals = sub.values.astype(float)
        vals = np.where(np.isfinite(vals) & (vals > -2.0) & (vals < 40.0), vals, np.nan)
        lon2d, lat2d = np.meshgrid(lon, lat)
        rs = BucketResampler(dbo_area, dka.from_array(lon2d, chunks="auto"),
                                       dka.from_array(lat2d, chunks="auto"))
        out = rs.get_average(dka.from_array(vals, chunks="auto"))
        return (out.compute() if hasattr(out, "compute") else out).astype(np.float32)

    span = (BACKFILL_START.isoformat(), (_need[-1].to_timestamp("M")).date().isoformat())
    gr = None
    for k in range(5):
        try:
            gr = earthaccess.search_data(short_name=SST_SHORT, temporal=span); break
        except Exception:
            _time.sleep(2 * (k + 1))
    if gr is None:
        raise RuntimeError(f"CMR search failed for {SST_SHORT}")

    want = set(str(p) for p in _need)
    by_month = {}
    for g in gr:
        m = _gran_start(g)[:7]                       # 'YYYY-MM'
        if m in want:
            by_month.setdefault(m, g)                # one granule per month
    months = sorted(by_month)
    print(f"{len(months)} of {len(_need)} requested months available from CMR")

    ny, nx = dbo_area.shape
    buf, times = np.full((len(months), ny, nx), np.nan, np.float32), []
    t0 = _time.time()
    for k, m in enumerate(months):
        try:
            f = earthaccess.open([by_month[m]], show_progress=False)[0]
            with xr.open_dataset(f, engine="h5netcdf", chunks={}) as ds:
                v = SST_VAR if SST_VAR in ds.variables else list(ds.data_vars)[0]
                buf[len(times)] = regrid_sst(ds[v])
            times.append(np.datetime64(m + "-01"))
        except Exception as e:
            print(f"  skip {m} ({type(e).__name__}: {str(e)[:70]})")
            continue
        if k % 6 == 0:
            print(f"  {k}/{len(months)} months  ({_time.time()-t0:.0f}s)")

    base = xr.open_dataarray(BASE_SST)
    new  = xr.DataArray(buf[:len(times)], dims=("time", "y", "x"),
                        coords={"time": np.array(times), "y": ys, "x": xs}, name=base.name)
    assert new.shape[1:] == base.shape[1:], f"grid mismatch {new.shape[1:]} vs {base.shape[1:]}"

    ext = xr.concat([new, base], dim="time").sortby("time")
    _u, _i = np.unique(ext.time.values, return_index=True)          # drop any duplicate months
    ext = ext.isel(time=np.sort(_i))
    ext.attrs = dict(base.attrs)
    ext.attrs["note"] = (f"{len(times)} months backfilled to {BACKFILL_START:%Y-%m} by "
                         "08_sea-surface-density.ipynb; same product, bbox and bucket-average regrid "
                         f"as the original cache ({BASE_SST.name}, which is unmodified)")
    ext.to_netcdf(EXT_SST, encoding={ext.name: {"zlib": True, "complevel": 4}})
    _t2 = pd.to_datetime(ext.time.values)
    print(f"\nwrote {EXT_SST}\n  {len(_t2)} months ({_t2[0]:%Y-%m} -> {_t2[-1]:%Y-%m}) "
          f"in {(_time.time()-t0)/60:.1f} min")
    print(f"  added {len(times)} months; original cache untouched")
    base.close(); ext.close()

42 of 42 requested months available from CMR
  0/42 months  (3s)
  6/42 months  (13s)
  12/42 months  (22s)
  18/42 months  (31s)
  24/42 months  (41s)
  30/42 months  (50s)
  36/42 months  (60s)

wrote ~/outputs_density/monthly_sst_ease2_2011on.nc
  175 months (2011-09 -> 2026-04) in 1.3 min
  added 42 months; original cache untouched


## Load the monthly stacks

Self-contained loader so this runs on a fresh kernel. It searches for the caches the other
notebooks already wrote, handling **both** the project layout (`<project>/density/outputs_density/…`)
and the **flat home-directory layout** where notebooks write straight into `~/outputs_*/`, with
a recursive search as a fallback. If it still misses, set `SST_CACHE` / `SSS_CACHE` / `SIG_CACHE` at
the top of the cell to explicit paths.

| file | written by |
|---|---|
| `density/outputs_density/monthly_sst_ease2.nc` | `08_sea-surface-density.ipynb` §"Stream + reproject monthly SST" |
| `salinity/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc` | `2_figures/figS04_oisss-sss-box-trends.ipynb` §4 |
| `density/outputs_density/sea_surface_density_sigma0_ease2_monthly.nc` | `08_sea-surface-density.ipynb` (optional — recomputed here if SST+SSS are available) |

σ₀ is recomputed from SST + SSS when both are present (it is a few seconds of `gsw`), because the
thermal/haline decomposition further down needs the temperature and salinity fields as well. If
only the cached σ₀ stack is found, the map still works and the decomposition section is skipped.

In [15]:
# --- manual overrides: set any of these to a path string to bypass the search entirely ---
SST_CACHE = None
SSS_CACHE = None
SIG_CACHE = None

_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd() / "density", Path.cwd().parent / "density",
          Path.home(), Path.home() / "ADD_YOUR_PATH", Path.home() / "ADD_YOUR_PATH" / "density"]  # add your path

def _find(*relglobs, deep=True):
    """First existing match for any root-relative pattern; falls back to a recursive search.

    Handles both the project layout (<project>/density/outputs_density/...) and the flat
    home-directory layout where the notebooks write straight into ~/outputs_*/.
    """
    roots = list(dict.fromkeys(_ROOTS))
    for root in roots:
        for g in relglobs:
            hits = sorted(root.glob(g))
            if hits:
                return hits[0]
    if deep:                                  # last resort: recurse, skipping dot-directories
        leaf = relglobs[0].rsplit("/", 1)[-1]
        for root in roots:
            if not root.exists():
                continue
            hits = sorted(h for h in root.rglob(leaf)
                          if not any(q.startswith(".") for q in h.parts))
            if hits:
                return hits[0]
    return None

p_sst = Path(SST_CACHE) if SST_CACHE else _find(
    "outputs_density/monthly_sst_ease2_2011on.nc",                      # backfilled (preferred)
    "density/outputs_density/monthly_sst_ease2_2011on.nc",
    "outputs_density/monthly_sst_ease2.nc",                             # flat home-directory layout
    "density/outputs_density/monthly_sst_ease2.nc")                     # repo layout
p_sss = Path(SSS_CACHE) if SSS_CACHE else _find(
    "outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc",          # flat home-directory layout
    "salinity/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc", # repo layout
    "outputs_longterm_oisss/monthly_sss_ease2_oisss_*.nc",
    "salinity/outputs_longterm_oisss/monthly_sss_ease2_oisss_*.nc")
p_sig = Path(SIG_CACHE) if SIG_CACHE else _find(
    "outputs_density/sea_surface_density_sigma0_ease2_monthly.nc",
    "density/outputs_density/sea_surface_density_sigma0_ease2_monthly.nc")
print(f"SST    cache: {p_sst}\nSSS    cache: {p_sss}\nsigma0 cache: {p_sig}")

HAVE_TS = (p_sst is not None and p_sst.exists()) and (p_sss is not None and p_sss.exists())

def _month_start(da):
    return da.assign_coords(time=pd.to_datetime(da.time.values).to_period("M").to_timestamp())

def _span(da):
    t = pd.to_datetime(da.time.values)
    return f"{len(t)} months ({t[0]:%Y-%m} -> {t[-1]:%Y-%m})"

if HAVE_TS:
    sst = _month_start(xr.open_dataarray(p_sst))
    sss = _month_start(xr.open_dataarray(p_sss))
    print(f"  SST {_span(sst)}\n  SSS {_span(sss)}")
    _s0, _q0 = pd.to_datetime(sst.time.values)[0], pd.to_datetime(sss.time.values)[0]
    if _s0 > _q0 + pd.DateOffset(months=1):
        print(f"  ** the SST stack starts {_s0:%Y-%m} but salinity reaches back to {_q0:%Y-%m} — "
              f"SST is what is truncating\n     the record. Run the backfill section above to "
              f"recover those {(_s0.year-_q0.year)*12 + _s0.month-_q0.month} months. **")
    assert (sss.sizes["y"], sss.sizes["x"]) == dbo_area.shape, f"SSS grid {(sss.sizes['y'], sss.sizes['x'])} != {dbo_area.shape}"
    assert (sst.sizes["y"], sst.sizes["x"]) == dbo_area.shape, f"SST grid {(sst.sizes['y'], sst.sizes['x'])} != {dbo_area.shape}"
    common = sst.time.to_index().intersection(sss.time.to_index())
    if len(common) == 0:
        raise RuntimeError("No overlapping months between the SST and SSS stacks.")
    sst_c = sst.sel(time=common).values.astype(np.float32)
    sss_c = sss.sel(time=common).values.astype(np.float32)
    SA   = gsw.SA_from_SP(sss_c, 0.0, lons2d, lats2d).astype(np.float32)   # practical -> absolute salinity
    CT   = gsw.CT_from_t(SA, sst_c, 0.0).astype(np.float32)                # in-situ -> conservative temperature
    sig0 = gsw.sigma0(SA, CT).astype(np.float32)                           # p = 0 dbar (surface water pressure)
    ctime = common
    print(f"recomputed σ₀ from SST+SSS: {len(ctime)} months "
          f"({ctime[0]:%Y-%m} -> {ctime[-1]:%Y-%m}), range {np.nanmin(sig0):.2f}..{np.nanmax(sig0):.2f} kg/m³")
elif p_sig is not None and p_sig.exists():
    _d   = _month_start(xr.open_dataarray(p_sig))
    sig0 = _d.values.astype(np.float32); ctime = _d.time.to_index()
    SA = CT = sst_c = sss_c = None
    print(f"loaded cached σ₀ only: {_span(_d)} — thermal/haline decomposition will be skipped")
    print(f"  its recorded inputs: {_d.attrs.get('inputs', '(not recorded)')}")
    if ctime[0] >= pd.Timestamp("2013-01-01"):
        print("\n  ** WARNING — this cache starts "
              f"{ctime[0]:%Y-%m}, not 2011-09. OISSS v3 monthly reaches back to 2011-09, so this is"
              "\n     almost certainly the OLD SMAP-based σ₀ stack that 08_sea-surface-density.ipynb"
              "\n     superseded. Using it costs ~3.5 years of record AND the thermal/haline"
              "\n     decomposition. Locate the OISSS SSS cache (or rebuild it: section 4 of"
              "\n     2_figures/figS04_oisss-sss-box-trends.ipynb) and re-run this cell. **")
    if p_sst is None:
        print("  (no SST cache found either)")
else:
    raise FileNotFoundError(
        "No inputs found. Needed: monthly_sst_ease2.nc (08_sea-surface-density.ipynb) and "
        "monthly_sss_ease2_oisss_*.nc (section 4 of 2_figures/figS04_oisss-sss-box-trends.ipynb), "
        "or at minimum sea_surface_density_sigma0_ease2_monthly.nc. Set SST_CACHE / SSS_CACHE / "
        f"SIG_CACHE above to explicit paths.\n  searched roots: {[str(r) for r in dict.fromkeys(_ROOTS)]}")

sigma_da = xr.DataArray(sig0, dims=("time","y","x"), coords={"time": ctime, "y": ys, "x": xs}, name="sigma0")

SST    cache: ~/outputs_density/monthly_sst_ease2_2011on.nc
SSS    cache: ~/outputs_longterm_oisss/monthly_sss_ease2_oisss_gauss.nc
sigma0 cache: ~/outputs_density/sea_surface_density_sigma0_ease2_monthly.nc
  SST 175 months (2011-09 -> 2026-04)
  SSS 168 months (2011-09 -> 2025-08)
recomputed σ₀ from SST+SSS: 167 months (2011-09 -> 2025-08), range 9.61..26.88 kg/m³


## Annual open-water composites, per pixel

One value per pixel per year: the **May–September** median (or mean) of the monthly field. The
median is the default because it matches the SST trend map and it keeps a single cloud- or
ice-edge-contaminated month from dragging a year.

A year only counts for a pixel if it has at least `MIN_SEASON_MONTHS` (3) valid months in the
window — otherwise a pixel that saw one late-September scene would contribute a "year" that is
really a single month, and the record's first year (OISSS starts 2011-09) and last year (incomplete
season) would sit at the two ends of the fit where they have the most leverage. Years that fail the
guard everywhere are dropped from the stack entirely.

In [16]:
def annual_growing(da, stat=ANNUAL_STAT):
    """May–Sep composite per year, but only where the year has >= MIN_SEASON_MONTHS valid months.

    The month guard matters at both ends of the record: OISSS starts 2011-09 (September only) and
    the current year is usually incomplete. Without it those partial seasons enter the trend fit as
    full-weight endpoints, which is exactly where a Theil-Sen slope has the most leverage.
    """
    grow = da.sel(time=da["time.month"].isin(GROW_MONTHS))
    g    = grow.groupby("time.year")
    comp = g.median("time") if stat == "median" else g.mean("time")
    nmon = grow.notnull().groupby("time.year").sum("time")
    return comp.where(nmon >= MIN_SEASON_MONTHS).astype(np.float32)

ann_sig = annual_growing(sigma_da)
_nyr_kept = int((np.isfinite(ann_sig.values).sum(axis=(1, 2)) > 0).sum())
_dropped  = [int(y) for y, k in zip(ann_sig["year"].values,
                                    np.isfinite(ann_sig.values).sum(axis=(1, 2))) if k == 0]
if _dropped:
    print(f"years dropped entirely (< {MIN_SEASON_MONTHS} valid May–Sep months anywhere): {_dropped}")
ann_sig = ann_sig.sel(year=[y for y in ann_sig["year"].values if int(y) not in _dropped])
years   = ann_sig["year"].values.astype(float)
nvalid  = np.isfinite(ann_sig.values).sum(axis=0)
fit_mask = nvalid >= MIN_YEARS

print(f"annual frames: {int(years.min())}–{int(years.max())} ({len(years)} years, {ANNUAL_STAT} of May–Sep)")
print(f"pixels with >= {MIN_YEARS} years: {fit_mask.sum():,} of {fit_mask.size:,} "
      f"({100*fit_mask.sum()/fit_mask.size:.1f}% of the grid)")

years dropped entirely (< 3 valid May–Sep months anywhere): [2011]
annual frames: 2012–2025 (14 years, median of May–Sep)
pixels with >= 8 years: 221,527 of 765,570 (28.9% of the grid)


## Per-pixel trend: Theil-Sen + AR(1)-adjusted Mann-Kendall

Identical statistics to the box panels, just applied to every pixel:

- **Theil-Sen slope** — the median of all pairwise slopes, so a single bad year cannot swing it.
- **Mann-Kendall τ**, with the sample size replaced by an **effective** sample size
  `n_eff = n(1−φ)/(1+φ)` where φ is the lag-1 autocorrelation of the residuals about a least-squares
  line. Year-to-year σ₀ is correlated, and without this correction the p-values are optimistic.
  `n_eff` is clipped to [4, n] so a negative φ cannot manufacture more independence than there are
  observations. φ is taken over consecutive *valid* years (gaps closed up), matching the scalar
  `_l1()` helper the other DBO notebooks use.

The loop version in `06_modis-sst-daily.ipynb` takes minutes; with only ~14 annual values per pixel the whole
thing vectorises over pixels and runs in seconds. The next cell spot-checks it against
`scipy.stats.theilslopes` / `kendalltau` on random pixels to confirm the fast path is doing the same
arithmetic.

In [18]:
def pixelwise_trend(arr, x, min_n=MIN_YEARS, chunk=200_000):
    """Vectorised Theil-Sen slope (units/decade if x is in years) + AR(1)-adjusted MK p, per pixel.

    arr : (n, ny, nx) annual stack, NaN where missing.   x : (n,) time in years.
    Returns (slope_per_decade, p_adj, n_used, phi) each (ny, nx).
    """
    n, ny, nx = arr.shape
    flat = arr.reshape(n, -1).astype(np.float64)
    cnt  = np.isfinite(flat).sum(axis=0)
    keep = np.where(cnt >= min_n)[0]
    slope = np.full(flat.shape[1], np.nan); padj = np.full(flat.shape[1], np.nan)
    phi_o = np.full(flat.shape[1], np.nan)
    i, j  = np.triu_indices(n, k=1)
    dx    = (x[j] - x[i])[:, None]
    t0    = _time.time()
    for s in range(0, keep.size, chunk):
        cols = keep[s:s+chunk]
        y    = flat[:, cols]                                   # (n, m)
        m    = np.isfinite(y)
        nn   = m.sum(axis=0).astype(float)

        # --- Theil-Sen: median of pairwise slopes ---
        dy   = y[j] - y[i]
        slope[cols] = np.nanmedian(dy / dx, axis=0)

        # --- Kendall tau-a (x strictly increasing => sign(dx) = +1) ---
        S    = np.nansum(np.sign(dy), axis=0)
        tau  = S / (nn * (nn - 1) / 2.0)

        # --- lag-1 autocorrelation of OLS residuals ---
        # Pack each pixel's valid years to the top of the column (stable sort keeps time order),
        # so lag-1 is taken over CONSECUTIVE VALID values — the same convention as the scalar
        # _l1() helpers in 06_modis-sst-daily.ipynb / 08_sea-surface-density.ipynb, which operate on y[valid].
        order = np.argsort(~m, axis=0, kind="stable")
        yc    = np.take_along_axis(y, order, axis=0)
        xc    = np.take_along_axis(np.broadcast_to(x[:, None], y.shape), order, axis=0)
        vc    = np.arange(n)[:, None] < nn[None, :]
        yc    = np.where(vc, yc, 0.0); xc = np.where(vc, xc, 0.0)
        Sx   = xc.sum(axis=0); Sy = yc.sum(axis=0)
        Sxx  = (xc * xc).sum(axis=0); Sxy = (xc * yc).sum(axis=0)
        den  = nn * Sxx - Sx * Sx
        b    = np.where(den != 0, (nn * Sxy - Sx * Sy) / np.where(den == 0, 1, den), 0.0)
        a    = (Sy - b * Sx) / nn
        r    = np.where(vc, yc - (b * xc + a), np.nan)
        r1, r0 = r[1:], r[:-1]
        pm   = np.isfinite(r1) & np.isfinite(r0)
        npair = pm.sum(axis=0).astype(float)
        r1z, r0z = np.where(pm, r1, 0.0), np.where(pm, r0, 0.0)
        m1, m0 = r1z.sum(axis=0) / np.maximum(npair, 1), r0z.sum(axis=0) / np.maximum(npair, 1)
        c1, c0 = np.where(pm, r1z - m1, 0.0), np.where(pm, r0z - m0, 0.0)
        num  = (c1 * c0).sum(axis=0)
        dnm  = np.sqrt((c1 * c1).sum(axis=0) * (c0 * c0).sum(axis=0))
        phi  = np.where((npair >= 5) & (dnm > 0), num / np.where(dnm == 0, 1, dnm), 0.0)
        phi  = np.clip(phi, -0.99, 0.99)
        phi_o[cols] = phi

        # --- effective n, MK z, two-sided p ---
        ne   = np.where(phi != 0, nn * (1 - phi) / (1 + phi), nn)
        ne   = np.clip(ne, 4.0, nn)                                  # never more independent than n
        ok   = ne > 4
        var  = 2 * (2 * ne + 5) / (9 * ne * (ne - 1))
        z    = np.where(ok & (var > 0), tau / np.sqrt(np.where(var > 0, var, 1)), np.nan)
        padj[cols] = 2 * (1 - norm.cdf(np.abs(z)))
        if keep.size > chunk:
            print(f"  {min(s+chunk, keep.size):,}/{keep.size:,} pixels  ({_time.time()-t0:.0f}s)")
    print(f"fit {keep.size:,} pixels in {_time.time()-t0:.1f}s")
    return (slope.reshape(ny, nx) * 10.0, padj.reshape(ny, nx),
            cnt.reshape(ny, nx), phi_o.reshape(ny, nx))

sig_slope, sig_p, sig_n, sig_phi = pixelwise_trend(ann_sig.values, years)
_f = np.isfinite(sig_slope)
print(f"σ₀ trend: median {np.nanmedian(sig_slope):+.3f} kg m⁻³ dec⁻¹ | "
      f"{100*np.nanmean(sig_slope[_f] < 0):.0f}% of fit pixels lightening | "
      f"{100*np.nanmean((sig_p[_f] < P_HATCH)):.0f}% with p < {P_HATCH}")

  200,000/221,527 pixels  (2s)
  221,527/221,527 pixels  (2s)
fit 221,527 pixels in 2.1s
σ₀ trend: median -0.192 kg m⁻³ dec⁻¹ | 77% of fit pixels lightening | 28% with p < 0.1


## Save the trend fields

In [24]:
out = xr.Dataset(
    {"sigma0_trend": (("y","x"), sig_slope.astype(np.float32)),
     "p_mk_ar1":     (("y","x"), sig_p.astype(np.float32)),
     "n_years":      (("y","x"), sig_n.astype(np.int16)),
     "phi_lag1":     (("y","x"), sig_phi.astype(np.float32))},
    coords={"y": ys, "x": xs},
    attrs={"title": "Per-pixel sea-surface potential density (sigma0) trend, DBO EASE-Grid 2.0 North 4 km",
           "units": "kg m-3 per decade",
           "method": (f"annual May-Sep {ANNUAL_STAT} composites {int(years.min())}-{int(years.max())}; "
                      f"Theil-Sen slope; AR(1)-adjusted Mann-Kendall p (n_eff clipped to [4, n]); "
                      f"pixels require >= {MIN_YEARS} annual values"),
           "inputs": "MODIS Aqua L3 monthly nighttime SST 4km + OISSS v3 multimission monthly SSS; TEOS-10 sigma0 at p=0 dbar",
           "grid": "EASE-Grid 2.0 North (EPSG:6931), 4 km, area_extent=(-3624000, 992000, 0, 4372000)",
           "source_notebook": "1_pull_data/08_sea-surface-density.ipynb"})

fn = OUT_DIR / "sigma0_trend_map_ease2.nc"
out.to_netcdf(fn, encoding={v: {"zlib": True, "complevel": 4} for v in out.data_vars})
print(f"saved {fn}")
print("figures:")
for p in sorted(OUT_DIR.glob("sigma0_trend_map*.png")):
    print("  ", p)

saved outputs_density/sigma0_trend_map_ease2.nc
figures:
   outputs_density/sigma0_trend_map_coverage.png
   outputs_density/sigma0_trend_map_decomposition.png
   outputs_density/sigma0_trend_map_ease2.png
   outputs_density/sigma0_trend_map_ease2_sigonly.png
